# Experiment: Centerline Cleanup and EDA

In [1]:
from pathlib import Path

import fiona
import geopandas as gpd
import pandas as pd
import polars as pl
from pyproj.exceptions import CRSError
from shapely.geometry import shape


In [ ]:
shp_path = Path('../data/raw/Centreline/toronto_centerline.shp')

try:
    gdf = gpd.read_file(shp_path, engine='fiona')
except CRSError:
    # Fallback when local PROJ DB is unavailable in the current kernel.
    with fiona.open(shp_path) as src:
        rows = []
        for feat in src:
            props = dict(feat['properties'])
            props['geometry'] = shape(feat['geometry']) if feat['geometry'] else None
            rows.append(props)

    gdf = gpd.GeoDataFrame(rows, geometry='geometry')
    prj_path = shp_path.with_suffix('.prj')
    if prj_path.exists():
        gdf = gdf.set_crs(prj_path.read_text().strip(), allow_override=True)

gdf.head()


In [ ]:
print(f'CRS: {gdf.crs}')
print(f'Rows, columns: {gdf.shape}')
print('\nGeometry types:')
print(gdf.geom_type.value_counts(dropna=False))
print('\nNumeric summary:')
print(gdf.describe().T)


In [ ]:
# Standardize column names and normalize placeholder missing values.

rename_map = {
    '_id1': 'source_row_id',
    'CENTREL2': 'centerline_id',
    'LINEAR_3': 'linear_name_id',
    'LINEAR_4': 'street_name_full',
    'LINEAR_5': 'street_name_full_legal',
    'ADDRESS6': 'address_l',
    'ADDRESS7': 'address_r',
    'PARITY_8': 'parity_l',
    'PARITY_9': 'parity_r',
    'LO_NUM_10': 'lo_num_l',
    'HI_NUM_11': 'hi_num_l',
    'LO_NUM_12': 'lo_num_r',
    'HI_NUM_13': 'hi_num_r',
    'BEGIN_A14': 'begin_addr_point_id_l',
    'END_ADD15': 'end_addr_point_id_l',
    'BEGIN_A16': 'begin_addr_point_id_r',
    'END_ADD17': 'end_addr_point_id_r',
    'BEGIN_A18': 'begin_addr_l',
    'END_ADD19': 'end_addr_l',
    'BEGIN_A20': 'begin_addr_r',
    'END_ADD21': 'end_addr_r',
    'LOW_NUM22': 'low_num_odd',
    'HIGH_NU23': 'high_num_odd',
    'LOW_NUM24': 'low_num_even',
    'HIGH_NU25': 'high_num_even',
    'LINEAR_26': 'street_name',
    'LINEAR_27': 'street_type',
    'LINEAR_28': 'street_direction',
    'LINEAR_29': 'street_name_desc',
    'LINEAR_30': 'street_label',
    'FROM_IN31': 'from_intersection_id',
    'TO_INTE32': 'to_intersection_id',
    'ONEWAY_33': 'oneway_code',
    'ONEWAY_34': 'oneway_desc',
    'FEATURE35': 'feature_code',
    'FEATURE36': 'feature_desc',
    'JURISDI37': 'jurisdiction',
    'CENTREL38': 'centerline_status',
    'OBJECTI39': 'objectid',
    'MI_PRIN40': 'mi_prinx',
}

gdf = gdf.rename(columns=rename_map)
text_cols = gdf.select_dtypes(include='object').columns
gdf[text_cols] = gdf[text_cols].replace('None', pd.NA)


In [ ]:
attr_df = pl.DataFrame(gdf.drop(columns='geometry').to_dict(orient='list'))
attr_df.head()


In [ ]:
# Null profiling after normalizing literal 'None' placeholders.

missing_summary = pd.DataFrame({
    'column': attr_df.columns,
    'null_count': [attr_df[col].null_count() for col in attr_df.columns],
})
missing_summary['null_pct'] = (missing_summary['null_count'] / len(attr_df) * 100).round(2)
missing_summary = missing_summary.sort_values(['null_count', 'column'], ascending=[False, True])
print(missing_summary.to_string(index=False))


In [ ]:
# ID uniqueness checks.

id_columns = ['centerline_id', 'source_row_id', 'objectid', 'mi_prinx']
dup_summary = pd.DataFrame([
    {
        'column': col,
        'duplicate_rows': int(gdf[col].duplicated().sum()),
        'unique_values': int(gdf[col].nunique(dropna=False)),
    }
    for col in id_columns
])
print(dup_summary.to_string(index=False))

dup_centerline_ids = attr_df.group_by('centerline_id').len().filter(pl.col('len') > 1)
print('\nDuplicate centerline_id groups:')
print(dup_centerline_ids)


In [ ]:
# Domain distribution and code-to-description checks.

feature_counts = attr_df.group_by('feature_desc').len().sort('len', descending=True)
oneway_counts = attr_df.group_by('oneway_desc').len().sort('len', descending=True)
parity_counts = attr_df.select([
    pl.col('parity_l').value_counts(sort=True).alias('parity_l_counts'),
    pl.col('parity_r').value_counts(sort=True).alias('parity_r_counts'),
])

print('Feature counts:')
print(feature_counts)
print('\nOne-way counts:')
print(oneway_counts)
print('\nOne-way code/description pairs:')
print(attr_df.group_by(['oneway_code', 'oneway_desc']).len().sort('len', descending=True))
print('\nFeature code/description pairs:')
print(attr_df.group_by(['feature_code', 'feature_desc']).len().sort('len', descending=True))
print('\nParity counts:')
print(parity_counts)


In [ ]:
# Address range presence, order, and label consistency.

address_summary = pd.DataFrame([
    {
        'side': 'left',
        'label_present': int(gdf['address_l'].notna().sum()),
        'label_contains_hyphen': int(gdf['address_l'].str.contains('-', regex=False, na=False).sum()),
        'numeric_range_present': int((gdf['lo_num_l'].notna() & gdf['hi_num_l'].notna()).sum()),
        'label_without_numeric': int((gdf['address_l'].notna() & ~(gdf['lo_num_l'].notna() & gdf['hi_num_l'].notna())).sum()),
        'numeric_without_label': int(((gdf['lo_num_l'].notna() & gdf['hi_num_l'].notna()) & gdf['address_l'].isna()).sum()),
        'reversed_numeric_range': int(((gdf['lo_num_l'].notna() & gdf['hi_num_l'].notna()) & (gdf['hi_num_l'] < gdf['lo_num_l'])).sum()),
        'equal_lo_hi': int(((gdf['lo_num_l'].notna() & gdf['hi_num_l'].notna()) & (gdf['hi_num_l'] == gdf['lo_num_l'])).sum()),
    },
    {
        'side': 'right',
        'label_present': int(gdf['address_r'].notna().sum()),
        'label_contains_hyphen': int(gdf['address_r'].str.contains('-', regex=False, na=False).sum()),
        'numeric_range_present': int((gdf['lo_num_r'].notna() & gdf['hi_num_r'].notna()).sum()),
        'label_without_numeric': int((gdf['address_r'].notna() & ~(gdf['lo_num_r'].notna() & gdf['hi_num_r'].notna())).sum()),
        'numeric_without_label': int(((gdf['lo_num_r'].notna() & gdf['hi_num_r'].notna()) & gdf['address_r'].isna()).sum()),
        'reversed_numeric_range': int(((gdf['lo_num_r'].notna() & gdf['hi_num_r'].notna()) & (gdf['hi_num_r'] < gdf['lo_num_r'])).sum()),
        'equal_lo_hi': int(((gdf['lo_num_r'].notna() & gdf['hi_num_r'].notna()) & (gdf['hi_num_r'] == gdf['lo_num_r'])).sum()),
    },
])
print(address_summary.to_string(index=False))


In [ ]:
# Compare begin/end address fields to the low/high range fields.

begin_end_checks = []
for side, begin_col, end_col, lo_col, hi_col in [
    ('left', 'begin_addr_l', 'end_addr_l', 'lo_num_l', 'hi_num_l'),
    ('right', 'begin_addr_r', 'end_addr_r', 'lo_num_r', 'hi_num_r'),
]:
    mask = gdf[[begin_col, end_col, lo_col, hi_col]].notna().all(axis=1)
    begin_vals = gdf.loc[mask, begin_col]
    end_vals = gdf.loc[mask, end_col]
    lo_vals = gdf.loc[mask, lo_col]
    hi_vals = gdf.loc[mask, hi_col]

    exact_mask = (begin_vals == lo_vals) & (end_vals == hi_vals)
    reversed_mask = ((begin_vals == hi_vals) & (end_vals == lo_vals)) & ~exact_mask
    exact_match = exact_mask.sum()
    reversed_match = reversed_mask.sum()
    other_mismatch = int(mask.sum() - exact_match - reversed_match)

    begin_end_checks.append({
        'side': side,
        'rows_checked': int(mask.sum()),
        'exact_match': int(exact_match),
        'reversed_match': int(reversed_match),
        'other_mismatch': other_mismatch,
    })

print(pd.DataFrame(begin_end_checks).to_string(index=False))


In [ ]:
# Parity QA, including mixed OE ranges.

parity_checks = []
for side, parity_col, lo_col, hi_col in [
    ('left', 'parity_l', 'lo_num_l', 'hi_num_l'),
    ('right', 'parity_r', 'lo_num_r', 'hi_num_r'),
]:
    mask = gdf[[parity_col, lo_col, hi_col]].notna().all(axis=1)
    parity = gdf.loc[mask, parity_col]
    lo_parity = (gdf.loc[mask, lo_col] % 2).astype(int)
    hi_parity = (gdf.loc[mask, hi_col] % 2).astype(int)

    parity_checks.append({
        'side': side,
        'rows_checked': int(mask.sum()),
        'bad_even': int(((parity == 'E') & ((lo_parity != 0) | (hi_parity != 0))).sum()),
        'bad_odd': int(((parity == 'O') & ((lo_parity != 1) | (hi_parity != 1))).sum()),
        'bad_oe': int(((parity == 'OE') & (lo_parity == hi_parity)).sum()),
        'range_present_with_N': int((parity == 'N').sum()),
        'unknown_parity_values': int((~parity.isin(['E', 'O', 'OE', 'N'])).sum()),
    })

print(pd.DataFrame(parity_checks).to_string(index=False))


In [ ]:
# Geometry QA on the raw linework. Do not buffer line geometries for repair.

geometry_summary = pd.Series({
    'null_geometry': int(gdf.geometry.isna().sum()),
    'empty_geometry': int(gdf.geometry.is_empty.sum()),
    'invalid_geometry': int((~gdf.geometry.is_valid).sum()),
})
print(geometry_summary)
print('\nGeometry types:')
print(gdf.geom_type.value_counts(dropna=False))

assert geometry_summary.eq(0).all(), 'Raw centreline geometry QA failed.'


In [ ]:
# Lightweight topology and duplication checks before graph construction.

same_node_mask = gdf['from_intersection_id'] == gdf['to_intersection_id']
geometry_wkb = gdf.geometry.to_wkb()

topology_summary = pd.Series({
    'same_from_to_intersection': int(same_node_mask.sum()),
    'duplicate_geometry_rows': int(geometry_wkb.duplicated().sum()),
    'missing_from_intersection_id': int(gdf['from_intersection_id'].isna().sum()),
    'missing_to_intersection_id': int(gdf['to_intersection_id'].isna().sum()),
})
print(topology_summary)

if same_node_mask.any():
    print('\nSample same-node segments:')
    print(
        gdf.loc[same_node_mask, ['centerline_id', 'street_label', 'feature_desc', 'from_intersection_id', 'to_intersection_id']]
        .head(10)
        .to_string(index=False)
    )


In [ ]:
# Convert CRS to metric for segment length QA.

gdf = gdf.to_crs(26917)
print(gdf.crs)


In [ ]:
# Segment length distribution and outlier checks.

gdf['length_m'] = gdf.geometry.length
length_summary = pd.Series({
    'min_m': gdf['length_m'].min(),
    'p01_m': gdf['length_m'].quantile(0.01),
    'median_m': gdf['length_m'].median(),
    'p99_m': gdf['length_m'].quantile(0.99),
    'max_m': gdf['length_m'].max(),
    'segments_lt_5m': int((gdf['length_m'] < 5).sum()),
    'segments_gt_1000m': int((gdf['length_m'] > 1000).sum()),
}).round(2)
print(length_summary)
print('\nLongest segments:')
print(gdf.nlargest(10, 'length_m')[['centerline_id', 'street_label', 'feature_desc', 'length_m']].to_string(index=False))


In [ ]:
# Send derived metrics back to Polars for grouped summaries.

attr_df = pl.DataFrame(gdf.drop(columns='geometry').to_dict(orient='list'))

length_by_type = (
    attr_df
    .group_by('feature_desc')
    .agg([
        pl.len().alias('segment_count'),
        pl.col('length_m').sum().alias('total_length_m'),
        pl.col('length_m').mean().alias('avg_length_m'),
    ])
    .sort('total_length_m', descending=True)
)

print(length_by_type)
